
<div style="padding:24px 26px;border-radius:18px;background:#fff5f1;border:2px solid #ee8b67;">
  <div style="font-size:31px;font-weight:800;color:#d94f2b;">
    🎯 Monte Carlo Control — One Connected Learning Notebook
  </div>
  <div style="margin-top:10px;font-size:17px;line-height:1.6;color:#333;">
    Every function you complete is used by the next function and eventually becomes the final
    <b>ε-greedy Monte Carlo Control</b> algorithm.
  </div>
</div>

<div style="margin-top:14px;padding:15px 17px;border-left:6px solid #d94f2b;background:#fffaf8;border-radius:10px;">
  <b>The story:</b>
  choose an action → generate an episode → compute its returns → update Q → repeat.
</div>

<div style="display:flex;gap:10px;flex-wrap:wrap;margin:16px 0;">
  <span style="padding:9px 13px;border-radius:999px;background:#eef7ff;color:#175a8a;font-weight:700;">1. ε-greedy action</span>
  <span style="padding:9px 13px;border-radius:999px;background:#f5efff;color:#6742a4;font-weight:700;">2. Episode</span>
  <span style="padding:9px 13px;border-radius:999px;background:#fff8df;color:#8a6900;font-weight:700;">3. Return G</span>
  <span style="padding:9px 13px;border-radius:999px;background:#effaf1;color:#2d7840;font-weight:700;">4. Q update</span>
  <span style="padding:9px 13px;border-radius:999px;background:#fff0f0;color:#a23838;font-weight:700;">5. Train</span>
</div>

\[
\boxed{
\text{choose action}
\rightarrow
\text{episode}
\rightarrow
G_t
\rightarrow
Q(s,a)
\rightarrow
\text{new ε-greedy behavior}
}
\]

<div style="padding:19px 21px;border-radius:15px;background:#eef7ff;border:1px solid #b9d9f2;">
  <div style="font-size:24px;font-weight:800;color:#175a8a;">🌍 0 — Our Gridworld</div>
  <div style="margin-top:9px;color:#333;line-height:1.65;">
    The environment is <b>deterministic</b>. States <b>0</b> and <b>15</b> are terminal.
    Every move gives reward <b>-1</b>, so a good policy reaches a terminal state quickly.
  </div>
</div>

In [4]:
import random

grid_size = 4
num_states = grid_size * grid_size
states = list(range(num_states))

actions = ['up', 'down', 'right', 'left']

terminal_states = [0, 15]
reward = -1
gamma = 1.0

random.seed(7)

In [5]:
def is_terminal(state):
    return state in terminal_states


def get_next_state(state, action):
    """Environment helper: deterministic transition (state, action) -> next_state."""

    if is_terminal(state):
        return state

    row, col = divmod(state, grid_size)

    if action == 'up' and row > 0:
        return state - grid_size
    if action == 'down' and row < grid_size - 1:
        return state + grid_size
    if action == 'right' and col < grid_size - 1:
        return state + 1
    if action == 'left' and col > 0:
        return state - 1

    return state


def make_q_table():
    """Initialize Q(s,a)=0 for every state-action pair."""
    return {
        state: {action: 0.0 for action in actions}
        for state in states
    }


def random_non_terminal_state():
    return random.choice([s for s in states if not is_terminal(s)])


def print_grid():
    for row in range(grid_size):
        print(" | ".join(f"{row * grid_size + col:2d}" for col in range(grid_size)))


print_grid()

 0 |  1 |  2 |  3
 4 |  5 |  6 |  7
 8 |  9 | 10 | 11
12 | 13 | 14 | 15


<div style="padding:14px 16px;background:#f7f7f7;border:1px solid #ddd;border-radius:11px;">
<b>Helper already implemented:</b> <code>greedy_action()</code> returns one action with the largest
current Q-value. If there is a tie, it randomly chooses between the tied best actions.
</div>

In [6]:
def greedy_action(Q, state):
    best_value = max(Q[state].values())

    candidates = [
        action
        for action in actions
        if Q[state][action] == best_value
    ]

    return random.choice(candidates)

<div style="padding:20px 22px;border-radius:16px;background:#effaf1;border:1px solid #bfe1c7;">
  <div style="font-size:24px;font-weight:800;color:#2d7840;">🎲 1 — Choose an ε-greedy action</div>
  <div style="margin-top:10px;line-height:1.65;color:#333;">
    This function is the <b>current policy</b> used by the agent.
  </div>
</div>

<br>

With probability \(1-\epsilon\), exploit:

\[
a = \arg\max_a Q(s,a)
\]

With probability \(\epsilon\), explore by choosing a random action.

<div style="margin-top:12px;padding:14px 16px;border-left:6px solid #2d7840;background:#f8fff9;border-radius:10px;">
<b>Your task:</b> complete only the two action assignments below. The structure is already given.
</div>

In [9]:
def choose_action(Q, state, epsilon):
    """
    Choose one action using an epsilon-greedy policy.

    Parameters
    ----------
    Q : dict
        Current Q-table.
        Example:
        Q[5] = {'up': -3.0, 'down': -4.0, 'right': -1.0, 'left': -2.0}

    state : int
        Current state.
        Example: 5

    epsilon : float
        Exploration probability.
        Example: 0.10

    Returns
    -------
    str
        One action string.

        Example return:
        'right'

    What to implement
    -----------------
    - Exploration: choose any action randomly.
    - Exploitation: choose greedy_action(Q, state).

    Why
    ---
    generate_episode() will call this function at every step.
    So this function is the agent's behavior policy during training.
    """

    if random.random() < epsilon:
        ##### CODE HERE #####
        action =  random.choice(actions)
        ##### FINISH HERE #####

    else:
        ##### CODE HERE #####
        max_val = max( Q[state].values())
        action = [key for key, val in Q[state].items() if val == max_val][0]
        ##### FINISH HERE #####

    return action

In [10]:
# ✅ Check Exercise 1

test_Q = make_q_table()
test_Q[5] = {'up': -3.0, 'down': -4.0, 'right': 10.0, 'left': -2.0}

for _ in range(20):
    assert choose_action(test_Q, 5, epsilon=0.0) == 'right'

for _ in range(20):
    assert choose_action(test_Q, 5, epsilon=1.0) in actions

print("✅ choose_action() works and is ready for generate_episode().")

✅ choose_action() works and is ready for generate_episode().


<div style="padding:20px 22px;border-radius:16px;background:#f5efff;border:1px solid #d7c6f2;">
  <div style="font-size:24px;font-weight:800;color:#6742a4;">🎬 2 — Generate one complete episode</div>
  <div style="margin-top:10px;line-height:1.65;color:#333;">
    Now we use the <b>choose_action()</b> function you just implemented.
  </div>
</div>

<br>

Example episode:

```python
[
    (5, 'left', -1),
    (4, 'up', -1)
]
```

<div style="margin-top:12px;padding:14px 16px;border-left:6px solid #6742a4;background:#fbf9ff;border-radius:10px;">
<b>Your task:</b> complete the missing lines inside the already-built loop:
choose the action, compute the next state, and record the experience.
</div>

In [17]:
def generate_episode(Q, epsilon, start_state=None, max_steps=200):
    """
    Generate ONE complete episode using choose_action().

    Parameters
    ----------
    Q : dict
        Current Q-table.

    epsilon : float
        Exploration probability used by choose_action().

    start_state : int or None
        Optional start state.
        Example: 5
        If None, choose a random non-terminal state.

    max_steps : int
        Safety limit while debugging.

    Returns
    -------
    list of tuples
        Each item is (state, action, reward).

        Example return:
        [
            (5, 'left', -1),
            (4, 'up', -1)
        ]

    What to implement
    -----------------
    Inside the loop:
      1. call choose_action(Q, state, epsilon),
      2. call get_next_state(state, action),
      3. append (state, action, reward).

    Why
    ---
    Monte Carlo learns from complete sampled experience.
    """

    episode = []

    if start_state is None:
        state = random_non_terminal_state()
    else:
        state = start_state

    for _ in range(max_steps):

        if is_terminal(state):
            break

        ##### CODE HERE #####
        action = choose_action(Q, state, epsilon)
        next_state = get_next_state(state, action)
        ##### FINISH HERE #####

        episode.append((state,action,reward))

        state = next_state

    return episode

In [18]:
# ✅ Check Exercise 2

test_Q = make_q_table()
test_Q[5]['left'] = 10.0
test_Q[4]['up'] = 10.0

episode = generate_episode(
    test_Q,
    epsilon=0.0,
    start_state=5,
    max_steps=10
)

expected = [
    (5, 'left', -1),
    (4, 'up', -1)
]

assert episode == expected, f"Expected {expected}, got {episode}"

print("✅ generate_episode() works.")
print("Episode:", episode)

✅ generate_episode() works.
Episode: [(5, 'left', -1), (4, 'up', -1)]


<div style="padding:20px 22px;border-radius:16px;background:#fff8df;border:1px solid #ead896;">
  <div style="font-size:24px;font-weight:800;color:#8a6900;">🧮 3 — Compute the Monte Carlo return \(G_t\)</div>
  <div style="margin-top:10px;line-height:1.65;color:#333;">
    We now take the episode produced by <b>generate_episode()</b> and calculate the future return
    from each step.
  </div>
</div>

<br>

\[
\boxed{G_t = R_{t+1} + \gamma G_{t+1}}
\]

For rewards `[-1, -1, -1]` and \(\gamma=1\):

```python
[-3.0, -2.0, -1.0]
```

<div style="margin-top:12px;padding:14px 16px;border-left:6px solid #8a6900;background:#fffdf5;border-radius:10px;">
<b>Your task:</b> the result list, \(G\), and backward loop are already initialized.
Only implement the recurrence and store the result.
</div>

In [25]:
def compute_returns(episode):
    """
    Compute G_t for every step in an episode.

    Parameters
    ----------
    episode : list of tuples
        Output from generate_episode().

        Example:
        [
            (5, 'right', -1),
            (6, 'down', -1),
            (10, 'down', -1)
        ]

    Returns
    -------
    list of float
        Return for every step.

        Example return:
        [-3.0, -2.0, -1.0]

    What to implement
    -----------------
    Walk backward and use:

        G = step_reward + gamma * G

    Then store G at returns[t].

    Why
    ---
    G_t is the Monte Carlo learning target.
    """

    returns = [0.0] * len(a)
    G = 0.0

    for t in range(len(episode) - 1, -1, -1):

        state, action, step_reward = episode[t]

        ##### CODE HERE #####
        G = G + step_reward
        returns[t] = G
        ##### FINISH HERE #####

    return returns

In [26]:
# ✅ Check Exercise 3

test_episode = [
    (5, 'right', -1),
    (6, 'down', -1),
    (10, 'down', -1)
]

returns = compute_returns(test_episode)

assert returns == [-3.0, -2.0, -1.0], returns

print("✅ compute_returns() works.")
print("Returns:", returns)

✅ compute_returns() works.
Returns: [-3.0, -2.0, -1.0]


<div style="padding:20px 22px;border-radius:16px;background:#effaf1;border:1px solid #bfe1c7;">
  <div style="font-size:24px;font-weight:800;color:#2d7840;">📈 4 — Update \(Q(s,a)\)</div>
  <div style="margin-top:10px;line-height:1.65;color:#333;">
    Now the episode and its returns update the quantity we actually want to learn:
    <b>Q-values</b>.
  </div>
</div>

<br>

\[
\boxed{
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha\left(G_t-Q(s,a)\right)
}
\]

<div style="margin-top:12px;padding:14px 16px;border-left:6px solid #2d7840;background:#f8fff9;border-radius:10px;">
<b>Your task:</b> the first-visit bookkeeping is already written.
You only implement the actual Q-update.
</div>

In [27]:
def update_q_first_visit(Q, episode, returns, alpha):
    """
    Update Q using first-visit Monte Carlo.

    Parameters
    ----------
    Q : dict
        Q-table, updated in place.

    episode : list of tuples
        Output from generate_episode().

    returns : list of float
        Output from compute_returns().

    alpha : float
        Learning rate.
        Example: 0.1

    Returns
    -------
    dict
        Updated Q-table.

        Example changed item:
        Q[5]['right'] = -0.4

    What to implement
    -----------------
    For the first occurrence of each (state, action):

        old_q = Q[state][action]
        Q[state][action] = old_q + alpha * (G - old_q)

    Why
    ---
    This is Monte Carlo policy evaluation using action values.
    """

    visited = set()

    for t, (state, action, step_reward) in enumerate(episode):

        pair = (state, action)

        if pair in visited:
            continue

        visited.add(pair)
        G = returns[t]
        old_q = Q[state][action]

        ##### CODE HERE #####
        Q[state][action] = old_q + alpha * (G - old_q)
        ##### FINISH HERE #####

    return Q

In [28]:
# ✅ Check Exercise 4

test_Q = make_q_table()

test_episode = [
    (5, 'right', -1),
    (6, 'left', -1),
    (5, 'right', -1),
]

test_returns = [-3.0, -2.0, -1.0]

update_q_first_visit(
    test_Q,
    test_episode,
    test_returns,
    alpha=0.5
)

assert abs(test_Q[5]['right'] - (-1.5)) < 1e-12
assert abs(test_Q[6]['left'] - (-1.0)) < 1e-12

print("✅ update_q_first_visit() works.")
print("Q(5,right):", test_Q[5]['right'])
print("Q(6,left):", test_Q[6]['left'])

✅ update_q_first_visit() works.
Q(5,right): -1.5
Q(6,left): -1.0


<div style="padding:20px 22px;border-radius:16px;background:#fff5f1;border:1px solid #efb49e;">
  <div style="font-size:24px;font-weight:800;color:#d94f2b;">🔗 Everything is now connected</div>
  <div style="margin-top:12px;line-height:1.8;color:#333;">
    <code>choose_action()</code> is used by <code>generate_episode()</code>.<br>
    <code>generate_episode()</code> produces the input for <code>compute_returns()</code>.<br>
    <code>compute_returns()</code> produces the target for <code>update_q_first_visit()</code>.<br>
    All four are used by the training loop below.
  </div>
</div>

<div style="padding:20px 22px;border-radius:16px;background:#fff0f0;border:1px solid #efbbbb;">
  <div style="font-size:24px;font-weight:800;color:#a23838;">🚀 5 — Assemble Monte Carlo Control</div>
  <div style="margin-top:10px;line-height:1.65;color:#333;">
    The final algorithm is just a loop connecting the functions you already built.
  </div>
</div>

<br>

For every training episode:

\[
\text{generate episode}
\rightarrow
\text{compute } G_t
\rightarrow
\text{update } Q
\]

The next episode then calls `choose_action()` using the newly updated Q-table.

<div style="margin-top:12px;padding:14px 16px;border-left:6px solid #a23838;background:#fff8f8;border-radius:10px;">
<b>Your task:</b> initialize Q, then connect your own previous functions inside the loop.
</div>

In [29]:
def monte_carlo_control(
    num_episodes=50_000,
    alpha=0.05,
    epsilon_start=0.30,
    epsilon_min=0.02
):
    """
    Train an epsilon-greedy Monte Carlo Control agent.

    Parameters
    ----------
    num_episodes : int
        Number of complete training episodes.

    alpha : float
        Learning rate for the Q-update.

    epsilon_start : float
        Exploration rate at the beginning.

    epsilon_min : float
        Minimum exploration rate near the end.

    Returns
    -------
    dict
        Learned Q-table.

        Example:
        Q[5] might look like:
        {
            'up': -2.1,
            'down': -4.0,
            'right': -4.2,
            'left': -2.0
        }

    What to implement
    -----------------
    Inside the loop:
      1. call generate_episode(Q, epsilon),
      2. call compute_returns(episode),
      3. call update_q_first_visit(Q, episode, returns, alpha).

    Why
    ---
    This is the complete Monte Carlo Control loop.
    """

    Q = make_q_table()

    for episode_number in range(num_episodes):

        fraction_remaining = 1 - (episode_number / num_episodes)

        epsilon = max(
            epsilon_min,
            epsilon_start * fraction_remaining
        )

        ##### CODE HERE #####
        episode = generate_episode(Q, epsilon)
        returns = compute_returns(episode)

        # Call your Q-update function here.
        update_q_first_visit(Q, episode, returns, alpha)

        ##### FINISH HERE #####

    return Q

<div style="padding:18px 20px;border-radius:14px;background:#f7f7f7;border:1px solid #ddd;">
  <div style="font-size:22px;font-weight:800;color:#444;">🏁 Run your complete algorithm</div>
  <div style="margin-top:8px;color:#444;">
    If every previous function works, this cell trains the full agent.
  </div>
</div>

In [30]:
Q = monte_carlo_control()

print("Learned Q-values for state 5:")
for action in actions:
    print(f"{action:>5}: {Q[5][action]:7.3f}")

Learned Q-values for state 5:
   up:  -2.179
 down:  -4.142
right:  -4.113
 left:  -2.006


In [31]:
def extract_greedy_policy(Q):
    return {
        state: greedy_action(Q, state)
        for state in states
        if not is_terminal(state)
    }


def print_policy(policy):
    arrows = {
        'up': '↑',
        'down': '↓',
        'right': '→',
        'left': '←',
    }

    for row in range(grid_size):
        row_items = []

        for col in range(grid_size):
            state = row * grid_size + col

            if is_terminal(state):
                row_items.append(" T ")
            else:
                row_items.append(f" {arrows[policy[state]]} ")

        print("|".join(row_items))


policy = extract_greedy_policy(Q)

print("\nFinal greedy policy:\n")
print_policy(policy)


Final greedy policy:

 T | ← | ← | ← 
 ↑ | ← | → | ↓ 
 ↑ | → | → | ↓ 
 ↑ | → | → | T 


<div style="padding:17px 19px;border-radius:14px;background:#eef7ff;border:1px solid #b9d9f2;">
  <b style="color:#175a8a;">Final sanity check:</b>
  an optimal action should reduce the distance to the nearest terminal state by one step.
  Multiple actions can be equally optimal.
</div>

In [32]:
def distance(state_a, state_b):
    r1, c1 = divmod(state_a, grid_size)
    r2, c2 = divmod(state_b, grid_size)
    return abs(r1 - r2) + abs(c1 - c2)


def distance_to_nearest_terminal(state):
    return min(distance(state, t) for t in terminal_states)


def is_shortest_path_action(state, action):
    next_state = get_next_state(state, action)

    return (
        distance_to_nearest_terminal(next_state)
        == distance_to_nearest_terminal(state) - 1
    )


correct = 0
total = 0

for state, action in policy.items():
    total += 1

    if is_shortest_path_action(state, action):
        correct += 1

print(f"Shortest-path greedy actions: {correct}/{total}")

if correct == total:
    print("🎉 Excellent — the learned greedy policy is optimal for every non-terminal state.")
elif correct >= total - 2:
    print("🟡 Very close. Monte Carlo is sampled; try more episodes if needed.")
else:
    print("🔍 Re-run the earlier checker cells and inspect the training loop.")

Shortest-path greedy actions: 14/14
🎉 Excellent — the learned greedy policy is optimal for every non-terminal state.


<div style="padding:22px 24px;border-radius:17px;background:#fff5f1;border:2px solid #efb49e;">
  <div style="font-size:25px;font-weight:800;color:#d94f2b;">🧠 What you built</div>

  <div style="margin-top:14px;line-height:1.8;color:#333;">
    <b>1.</b> <code>choose_action()</code> — your ε-greedy policy<br>
    <b>2.</b> <code>generate_episode()</code> — collect a complete trajectory<br>
    <b>3.</b> <code>compute_returns()</code> — calculate \(G_t\)<br>
    <b>4.</b> <code>update_q_first_visit()</code> — learn Q-values<br>
    <b>5.</b> <code>monte_carlo_control()</code> — connect everything
  </div>

  <div style="margin-top:16px;padding:13px 15px;background:white;border-radius:11px;border:1px solid #ddd;">
    <b>Next:</b> Temporal-Difference learning asks:
    <i>“Why wait until the episode ends before updating?”</i>
  </div>
</div>